<a href="https://colab.research.google.com/github/m9ts/AD-temp-inst/blob/main/TrabalhoII_temp_inst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho II: Algoritmos de regressão supervisionada na classificação de dados meteorológicos
**Disciplina:** Análise de Dados

**Objetivo:** Realizar a limpeza, tratamento e preparação de um conjunto de dados climáticos para treinar e avaliar múltiplos algoritmos de classificação (Regressão Logística, Árvores de Decisão, Random Forest, SVM, KNN, entre outros).

**Integrantes:** João Mizuno e Mateus Gois



---




# Preparação do ambiente

In [167]:
!git clone https://github.com/m9ts/AD-temp-inst.git

fatal: destination path 'AD-temp-inst' already exists and is not an empty directory.


In [168]:
path = "/content/drive/MyDrive/AD/Atividade_II/database_temp_inst.csv"

In [169]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

In [170]:
pd.set_option('display.float_format','{:.2f}'.format)

In [171]:
df_temp_inst = pd.read_csv(path, sep = ';')

In [172]:
df_temp_inst.head(10)

,data,hora,pressao,radiacao,temp_inst,pto_orvalho_inst,umid_inst,vento_vel,Unnamed: 8
0,2013-12-03,0,949.30,-3.54,22.30,21.00,92.00,2.30,NaN
1,2013-12-03,100,949.70,-3.54,22.10,20.60,91.00,2.10,NaN
2,2013-12-03,200,949.40,-3.54,22.10,20.00,88.00,1.30,NaN
3,2013-12-03,300,949.10,-3.54,21.90,20.20,90.00,2.30,NaN
4,2013-12-03,400,948.60,-3.54,21.40,19.80,91.00,1.50,NaN
5,2013-12-03,500,947.80,-3.54,21.20,19.80,92.00,2.00,NaN
6,2013-12-03,600,947.30,-3.54,20.80,19.40,92.00,1.80,NaN
7,2013-12-03,700,947.70,-3.54,20.40,19.60,95.00,1.70,NaN
8,2013-12-03,800,948.30,-3.44,20.40,19.70,95.00,1.80,NaN
9,2013-12-03,900,949.20,38.05,20.60,19.80,95.00,1.90,NaN


In [173]:
print(f'O dataset possui {df_temp_inst.shape[0]} entradas e {df_temp_inst.shape[1]} variáveis.')
print(f'Os tipos de dados são:\n{df_temp_inst.dtypes}')

O dataset possui 83232 entradas e 9 variáveis.
Os tipos de dados são:
data                 object
hora                  int64
pressao             float64
radiacao            float64
temp_inst           float64
pto_orvalho_inst    float64
umid_inst           float64
vento_vel           float64
Unnamed: 8          float64
dtype: object




---



# Remoção de colunas inutilizadas

In [174]:
# Coluna "Unnamed: 8" removida do dataset
colunas_para_remover = [col for col in df_temp_inst.columns if 'Unnamed' in col]
df_temp_inst = df_temp_inst.drop(columns=colunas_para_remover, errors='ignore')

print('Colunas restantes:', df_temp_inst.columns.tolist())

Colunas restantes: ['data', 'hora', 'pressao', 'radiacao', 'temp_inst', 'pto_orvalho_inst', 'umid_inst', 'vento_vel']


# Tratamento de valores ausentes

In [175]:
print('Valores nulos antes do tratamento:')
print(df_temp_inst.isnull().sum())

df_temp_inst = df_temp_inst.dropna(subset=['temp_inst', 'pressao', 'radiacao', 'umid_inst'])

print('\nValores nulos após o tratamento:')
print(df_temp_inst.isnull().sum())

Valores nulos antes do tratamento:
data                   0
hora                   0
pressao             5279
radiacao            5279
temp_inst           5279
pto_orvalho_inst    5284
umid_inst           5285
vento_vel           5279
dtype: int64

Valores nulos após o tratamento:
data                0
hora                0
pressao             0
radiacao            0
temp_inst           0
pto_orvalho_inst    2
umid_inst           0
vento_vel           0
dtype: int64


In [176]:
# Dados faltantes
print(f'As variáveis com mais dados faltantes, em porcentagem(%), são:\n')
((df_temp_inst.isnull().sum() / df_temp_inst.shape[0])*100).sort_values(ascending=False)

As variáveis com mais dados faltantes, em porcentagem(%), são:



,0
pto_orvalho_inst,0.00
data,0.00
pressao,0.00
hora,0.00
radiacao,0.00
temp_inst,0.00
umid_inst,0.00
vento_vel,0.00


# Engenharia de Recursos
Modelos de Machine Learning não interpretam o formato de data/texto diretamente. Portanto, convertemos a coluna `data` para o tipo `datetime` e extraímos variáveis numéricas sazonais (ano, mês, dia, dia da semana) que ajudarão os classificadores a entender padrões temporais.

In [177]:
# Garantindo que a coluna de data está no formato correto de datetime
df_temp_inst['data'] = pd.to_datetime(df_temp_inst['data'])

df_temp_inst['ano'] = df_temp_inst['data'].dt.year
df_temp_inst['mes'] = df_temp_inst['data'].dt.month
df_temp_inst['dia'] = df_temp_inst['data'].dt.day
df_temp_inst['dia_semana'] = df_temp_inst['data'].dt.dayofweek

In [178]:
df_temp_inst

,data,hora,pressao,radiacao,temp_inst,pto_orvalho_inst,umid_inst,vento_vel,ano,mes,dia,dia_semana
0,2013-12-03,0,949.30,-3.54,22.30,21.00,92.00,2.30,2013,12,3,1
1,2013-12-03,100,949.70,-3.54,22.10,20.60,91.00,2.10,2013,12,3,1
2,2013-12-03,200,949.40,-3.54,22.10,20.00,88.00,1.30,2013,12,3,1
3,2013-12-03,300,949.10,-3.54,21.90,20.20,90.00,2.30,2013,12,3,1
4,2013-12-03,400,948.60,-3.54,21.40,19.80,91.00,1.50,2013,12,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...
83227,2023-06-01,1900,954.20,1195.46,24.90,15.80,57.00,1.60,2023,6,1,3
83228,2023-06-01,2000,954.50,462.95,23.10,15.80,64.00,0.80,2023,6,1,3
83229,2023-06-01,2100,954.70,8.56,20.00,15.40,75.00,1.20,2023,6,1,3
83230,2023-06-01,2200,955.20,-3.54,19.30,15.40,78.00,1.30,2023,6,1,3


## Padronização e formatação da coluna de horas
Como a coluna original de horas foi interpretada como numérica, os zeros à esquerda foram omitidos (ex: `0` em vez de `0000`, `100` em vez de `0100`). Para corrigir isto, preenchemos os valores para que tenham sempre 4 dígitos, criamos uma coluna numérica (`hora_num`) para os algoritmos de Machine Learning e formatamos a coluna original no padrão textual legível `HH:MM`.

In [179]:
df_temp_inst['hora'] = df_temp_inst['hora'].astype(str).str.replace(r'\D', '', regex=True)

df_temp_inst['hora'] = df_temp_inst['hora'].str.zfill(4)

df_temp_inst['hora'] = df_temp_inst['hora'].str[:2] + ':' + df_temp_inst['hora'].str[2:]

df_temp_inst.head()

,data,hora,pressao,radiacao,temp_inst,pto_orvalho_inst,umid_inst,vento_vel,ano,mes,dia,dia_semana
0,2013-12-03,00:00,949.30,-3.54,22.30,21.00,92.00,2.30,2013,12,3,1
1,2013-12-03,01:00,949.70,-3.54,22.10,20.60,91.00,2.10,2013,12,3,1
2,2013-12-03,02:00,949.40,-3.54,22.10,20.00,88.00,1.30,2013,12,3,1
3,2013-12-03,03:00,949.10,-3.54,21.90,20.20,90.00,2.30,2013,12,3,1
4,2013-12-03,04:00,948.60,-3.54,21.40,19.80,91.00,1.50,2013,12,3,1


## Definição da variável alvo
Como o objetivo do trabalho é aplicar algoritmos de classificação, precisamos de uma variável dependente ($Y$) categórica. Criaremos uma regra de negócio baseada na temperatura instantânea (`temp_inst`): registros com temperatura acima de 25°C serão classificados como **1 (Clima Quente)**, e registros iguais ou abaixo de 25°C serão classificados como **0 (Clima Agradável/Frio)**. Em seguida, verificamos o balanceamento dessas duas classes.

In [180]:
limite_temperatura = 25.0

df_temp_inst['target_class'] = (df_temp_inst['temp_inst'] > limite_temperatura).astype(int)

print('Quantidade de registros por classe:')
print(df_temp_inst['target_class'].value_counts())

print('\nPercentual de distribuição das classes:')
print(df_temp_inst['target_class'].value_counts(normalize=True) * 100)

Quantidade de registros por classe:
target_class
0    46430
1    31517
Name: count, dtype: int64

Percentual de distribuição das classes:
target_class
0   59.57
1   40.43
Name: proportion, dtype: float64


## Separação de atributos (X) e alvo (y) e divisão treino/teste
Separamos as variáveis preditoras ($X$) da nossa variável alvo ($Y$). Para evitar que o modelo "cole" a resposta, removemos a coluna original `temp_inst` que deu origem ao target, além de colunas puramente textuais (como `data`). Dividimos o conjunto de dados na proporção de 80% para treinamento dos algoritmos e 20% para a validação dos testes.

In [181]:
X = df_temp_inst.drop(columns=['target_class', 'temp_inst', 'data', 'hora', 'data_hora'], errors='ignore')
y = df_temp_inst['target_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Formato dos dados de treino (X_train): {X_train.shape}")
print(f"Formato dos dados de teste (X_test): {X_test.shape}")

Formato dos dados de treino (X_train): (62357, 9)
Formato dos dados de teste (X_test): (15590, 9)


# Escalonamento e normalização dos dados
Algoritmos de classificação como Regressão Logística, SVM e KNN são sensíveis à escala das variáveis (por exemplo, a pressão tem valores na casa de 900, enquanto o vento fica na casa de 1 a 5). Utilizamos o `StandardScaler` para padronizar os recursos com média 0 e variância 1, garantindo que todas as variáveis tenham o mesmo peso no modelo.

In [186]:
scaler = StandardScaler()

# Ajustando o escalonador com os dados de treino e transformando-os
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Exemplo dos dados de treino escalonados (primeira linha):")
print(X_train_scaled[0])

Exemplo dos dados de treino escalonados (primeira linha):
[-7.63481782e-02 -7.60657362e-01  8.89880837e-01  7.90546366e-01
 -5.78276040e-01 -1.14533223e+00  1.57676973e+00  1.45531620e-01
 -2.88422360e-04]
